## 1) Importar bibliotecas

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 2) Importar base de dados

In [ ]:
df_train = pd.read_csv("./dados/train_linear.csv")
df_test = pd.read_csv("./dados/test_linear.csv")

In [ ]:
df_train.head(2)

In [ ]:
df_test.head(2)

In [ ]:
fig, ax = plt.subplots()

ax.scatter(x = df_train["x"], y = df_train["y"], c = df_train["class"], edgecolors = "#000")
ax.set_title("Distribuição dos pontos classificados.")

plt.show()

Aleatorizar <code>df_train</code> porque é sempre bom suspeitar de ordenação.

In [ ]:
df_train_resampled = df_train.sample(frac=1, random_state=1).reset_index(drop = True)

X = df_train_resampled[["x", "y"]].values
y = df_train_resampled["class"].values

print(X)
print(y)

## 3) Criando rede neural

### 3.1) Definindo rede neural:

<p>A princípio, recomenda-se que a primeira camada de <code>Input</code> seja o mesmo que a dimensão y da base de dados <code>X</code>.

Layers que serão definidos nas <q>camadas ocultas</q> poderão ter um quantidade de <code>units</code> variável de acordo com o problema, porém, sua função de ativação deve ser definida segundo o indicado em <code>tensorflow.ipynb</code>.

Na última camada - que também é um <code>Dense</code> -, a quantidade em <code>units</code> deve ser a mesma que a quantidade de classes a serem definidas. Por sua vez, a função de ativação desta etapa deve ser coerente com o que se está tentando demonstrar.</p>

In [ ]:
X.shape

In [ ]:
model = tf.keras.Sequential([
    # Isto vem de X.shape (4000, 2) -> 2:
    tf.keras.Input(shape=(2,)),
    # units é testável, mas ReLU é recomendável para RNNs no geral:
    tf.keras.layers.Dense(units = 4, activation = "relu"),
    # units = 2 porque a quantidade de classes são 2 e
    # activation = "sigmoid" porque estamos lidando com binário (mas poderia ser softmax):
    tf.keras.layers.Dense(units = 2, activation = "sigmoid")
])

model.summary()

### 3.2) Definindo parâmetros otimizadores e de aprendizado:

1. **Definição do Otimizador**: o otimizador é o algoritmo usado para atualizar os pesos da rede para minimizar a função de custo (ou perda). Alguns exemplos de otimizadores incluem Gradiente Descendente Estocástico (SGD), Adam, e RMSprop.

2. **Definição da Função de Custo**: A função de custo (ou função de perda) mede o quão bem o modelo está performando nos dados de treinamento. O objetivo é minimizar essa função para "treinar" o modelo. Exemplos de funções de custo incluem entropia cruzada categórica para classificação e erro quadrático médio para regressão.

3. **Definição das Métricas**: As métricas são usadas para monitorar o desempenho do modelo durante o treinamento e a validação. Elas não são usadas durante o treinamento para atualizar o modelo, mas são úteis para avaliar o desempenho do modelo. Exemplos de métricas incluem acurácia, precisão, recall, entre outras.


In [ ]:
# Por padrão, Adam é sempre o mais utilizado:
optimizer = tf.keras.optimizers.Adam()

loss = tf.keras.losses.SparseCategoricalCrossentropy()

metric = tf.keras.metrics.SparseCategoricalAccuracy()

model.compile(
    optimizer = optimizer,
    loss = loss,
    metrics = [metric]
)

### 3.3) Definindo parada:

<p>Definir e configurar uma parada é importante não só porque pode economizar recursos, como também pode ajudar a determinar onde o modelo está realmente parando de progredir - se é que isso ocorre.</p>

In [ ]:
model_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta = 10**(-3),
    patience = 5,
    verbose = 1
)

history = model.fit(
    X, y,
    epochs = 100,
    batch_size = 2**5,
    validation_split = 0.2,
    callbacks = [model_stopping]
)

### 3.4) Mostrando evolução:

In [ ]:
fig, ax = plt.subplots()

ax.plot(history.epoch, history.history["loss"], "C0", label = "loss")
ax.plot(history.epoch, history.history["val_loss"], "C1", label = "val_loss")

ax.set_title("Linha do tempo (epochs) do valor loss.")

plt.legend()
plt.show()

## 4) Testando modelo

In [ ]:
X_test = df_test[["x", "y"]].values
y_test = df_test[["class"]].values

prediction = model.predict(
    X_test
)

print(prediction)